# GĐ2 ngày 2: dựng môi trường AntiDeepfake và chạy toy example

**Mục tiêu:** EER đúng **6,11%**, log không có cảnh báo nạp trọng số.

**Trước khi chạy (chỉ làm lần đầu):**
1. Runtime → Change runtime type → **T4 GPU**.
2. Nếu repo private: thanh bên trái → biểu tượng chìa khoá (Secrets) → thêm `GITHUB_TOKEN`, bật *Notebook access*.

**Chạy:** Runtime → Run all. Ô dựng môi trường mất khoảng 10–20 phút ở lần đầu mỗi phiên.

| Ở đâu | Chứa gì | Còn sau khi hết phiên? |
|---|---|---|
| `/content/work` | Môi trường, code, kết quả tạm | Không |
| Drive `orpheus/cache` | Checkpoint, dữ liệu tải về | Có |
| Drive `orpheus/runs/<tên>_<thời gian>` | Log, file điểm, phiên bản | Có |

## 1. Cấu hình (chỉ sửa ô này)

In [ ]:
REPO_URL   = "https://github.com/beebapcay/orpheus-voice.git"  # repo của nhóm
BRANCH     = "main"
BASE       = "/content/work"                    # máy Colab, mất khi hết phiên
DRIVE_ROOT = "/content/drive/MyDrive/orpheus"   # lưu lâu dài trên Drive
RUN_NAME   = "gd2_toy"

## 2. Kiểm tra GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"

## 3. Nạp Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
CACHE = f"{DRIVE_ROOT}/cache"
RUNS  = f"{DRIVE_ROOT}/runs"
for d in (BASE, CACHE, RUNS):
    os.makedirs(d, exist_ok=True)
print("cache:", CACHE)
print("runs :", RUNS)

## 4. Lấy code của nhóm từ GitHub

Lần đầu thì clone, các lần sau thì pull bản mới nhất. Token lấy từ Colab Secrets, không nằm trong lệnh git, không bị in ra khi lỗi.

In [ ]:
import os, base64, subprocess
REPO_DIR = f"{BASE}/orpheus-voice"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # repo công khai thì không cần token

# Token đi qua biến môi trường của git (không nằm trong lệnh),
# nên không bị in ra khi lệnh lỗi và không lưu vào .git/config.
env = os.environ.copy()
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic}",
    })

def git(*args):
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).strip()
        if token:
            msg = msg.replace(token, "***")
        raise RuntimeError(f"git {args[0]} lỗi (mã {r.returncode}):\n{msg}")
    return r.stdout.strip()

if not os.path.isdir(f"{REPO_DIR}/.git"):
    git("clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR)
else:
    git("-C", REPO_DIR, "pull", "-q", "origin", BRANCH)

COMMIT = git("-C", REPO_DIR, "log", "-1", "--format=%h %s")
print("Code của nhóm:", COMMIT)

## 5. Dựng môi trường

Python 3.9 qua micromamba, torch 2.6.0 (CUDA 11.8), fairseq và AntiDeepfake ở commit cố định. Chạy lại ô này thì bước nào đã xong sẽ được bỏ qua.

Nếu lỗi: chạy ô "Xem log" ở cuối notebook và gửi 50 dòng cuối.

In [ ]:
!CACHE_DIR="{CACHE}" bash "{REPO_DIR}/src/setup_antideepfake.sh" "{BASE}" env

## 6. Chạy toy example

Cần thấy:
- `Nạp trọng số: không có cảnh báo`
- `evaluation.py`: `eer` = **0.0611**
- `evaluate.py`: dòng `Pooled` có `eer_%` = **6.1101**

In [ ]:
!CACHE_DIR="{CACHE}" bash "{REPO_DIR}/src/setup_antideepfake.sh" "{BASE}" toy

## 7. Lưu kết quả về Drive

In [ ]:
import os, shutil, datetime, subprocess
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
OUT = f"{RUNS}/{RUN_NAME}_{stamp}"
os.makedirs(OUT, exist_ok=True)

exp = f"{BASE}/Log/exps/exp_mms_300m_eval_toy_mms_300m"
files = [f"{BASE}/{f}" for f in ("setup.log", "toy_inference.log", "versions.txt", "pip-freeze.txt")]
files += [f"{exp}/evaluation_score.csv", f"{exp}/metrics.csv"]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, OUT)
    else:
        print("Không có:", f)

with open(f"{OUT}/run-info.txt", "w") as fh:
    fh.write(f"code nhóm: {COMMIT}\n")
    fh.write(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                            capture_output=True, text=True).stdout)
print("Đã lưu vào:", OUT)
print(sorted(os.listdir(OUT)))

## Xem log khi có lỗi

In [ ]:
!tail -n 50 "{BASE}/setup.log"